# Gemma 3 (4B) — ZERO-SHOT EXPERIMENT B ONLY (with explicit density definition, Colab)

This notebook runs Experiment B only (test image + YOLO count) — for Experiment A, use `Gemma_ExperimentA_ZeroShot_DensityDef.ipynb`.

Same model, same technical setup as the original Gemma zero-shot Experiment B notebook. The only change: the system prompt's density-labels section now defines what Low/Medium/High mean visually. Outputs save with a `_densitydef` suffix so the original results aren't overwritten.

## 1. Authenticate with Hugging Face

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

## 2. Mount Drive & confirm input data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from zipfile import ZipFile

zip_path = "/content/drive/MyDrive/CrowdDensityResearch/Phase5_Input.zip"

if not os.path.exists("/content/Phase5_Input"):
    with ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("/content/")

ROOT = "/content/Phase5_Input"
TEST_DIR = f"{ROOT}/test/images"
print("TEST_DIR exists:", os.path.exists(TEST_DIR))

## 3. Back up any existing zero-shot B results before overwriting

In [ ]:
import shutil

RESULT_DIR = "/content/drive/MyDrive/CrowdDensityResearch/VLM_Results"
os.makedirs(RESULT_DIR, exist_ok=True)

for fname in ["gemma_experiment_B_zeroshot_densitydef.csv", "gemma_experiment_B_zeroshot_densitydef_evaluated.csv"]:
    src = os.path.join(RESULT_DIR, fname)
    if os.path.exists(src):
        backup = os.path.join(RESULT_DIR, fname.replace(".csv", "_backup.csv"))
        if not os.path.exists(backup):
            shutil.copy2(src, backup)
            print(f"Backed up {fname} -> {os.path.basename(backup)}")
        else:
            print(f"Backup already exists for {fname}, skipping.")
    else:
        print(f"No existing {fname} found — nothing to back up.")

## 4. Install packages

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

## 5. Load the model
Same confirmed-working setup as zero-shot Experiment A — 4-bit quantized, single GPU.

In [ ]:
import torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig

MODEL_NAME = "google/gemma-3-4b-it"

print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs visible:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}, {props.total_memory / 1024**3:.2f} GiB")

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_NAME)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

print("Loading model (4-bit quantized)...")
model = Gemma3ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()

print("✅ Model loaded successfully!")
print()
print("--- Memory diagnostic ---")
if hasattr(model, "get_memory_footprint"):
    print(f"Model memory footprint: {model.get_memory_footprint() / 1024**3:.2f} GiB")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i} allocated: {torch.cuda.memory_allocated(i) / 1024**3:.2f} GiB")

## 6. Patch the vision tower to process images one at a time
Kept as a safety net — with 1 image per prompt this loop runs once, harmless.

In [ ]:
import types
from transformers.modeling_outputs import BaseModelOutputWithPooling


def _get_image_features_one_at_a_time(self, pixel_values, **kwargs):
    """Drop-in replacement for Gemma3Model.get_image_features(). Processes
    images through the vision tower one at a time instead of batched."""
    all_hidden_states = []
    for i in range(pixel_values.shape[0]):
        single_image = pixel_values[i : i + 1]
        vision_out = self.vision_tower(pixel_values=single_image, **kwargs)
        all_hidden_states.append(vision_out.last_hidden_state)
        del vision_out
        torch.cuda.empty_cache()

    last_hidden_state = torch.cat(all_hidden_states, dim=0)
    pooler_output = self.multi_modal_projector(last_hidden_state)
    return BaseModelOutputWithPooling(
        last_hidden_state=last_hidden_state,
        pooler_output=pooler_output,
    )


model.model.get_image_features = types.MethodType(_get_image_features_one_at_a_time, model.model)
print("✅ Vision tower patched to process images one at a time.")

## 7. Helpers

In [ ]:
from PIL import Image

def load_image(path, max_size=512):
    """Load an image and resize it to reduce memory usage."""
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_size, max_size))
    return img

DENSITY_TEXT = {
    0: "Low",
    1: "Medium",
    2: "High"
}

## 8. Zero-shot Experiment A's exact prompt (reference only — not re-run here)
Kept verbatim so Section 10's diff check can prove B differs from A only where intended.

In [ ]:
def build_messages_zeroshot_A(test_image):
    """Gemma 3 conversation for one test image — zero-shot Experiment A (reference only)."""
    conversation = []

    conversation.append({
        "role": "system",
        "content": [{
            "type": "text",
            "text": """You are an expert in crowd counting and crowd density estimation.

You will receive one image.

Predict:

1. The EXACT crowd count.
2. The crowd density.

Density labels — judge by how visually packed the crowd appears, not by exact count:

0 = Low: People are clearly separated, with visible space between individuals. Each person is easy to distinguish on their own.

1 = Medium: People are moderately close together. Some overlap between individuals, but gaps and open space are still visible.

2 = High: People are tightly packed with little or no visible space between them. Individuals often overlap and may be hard to distinguish separately.

Return ONLY:

Count: <integer>
Density: <0/1/2>

Do not explain.
Do not estimate.
Do not output ranges.
Return exactly one integer count.
"""
        }]
    })

    conversation.append({
        "role": "user",
        "content": [
            {"type": "image", "image": load_image(test_image)},
            {"type": "text", "text": """TEST IMAGE

Count every visible person exactly once.

Return ONLY

Count: <integer>

Density: <0/1/2>
"""}
        ]
    })

    return conversation

## 9. Zero-shot Experiment B — A's template + one YOLO block

In [ ]:
def build_messages_zeroshot_B(test_image, yolo_count):
    """
    Gemma 3 conversation for one test image — zero-shot Experiment B.
    Identical to build_messages_zeroshot_A() above except for the YOLO block marked below.
    """
    conversation = []

    conversation.append({
        "role": "system",
        "content": [{
            "type": "text",
            "text": """You are an expert in crowd counting and crowd density estimation.

You will receive one image.

Predict:

1. The EXACT crowd count.
2. The crowd density.

Density labels — judge by how visually packed the crowd appears, not by exact count:

0 = Low: People are clearly separated, with visible space between individuals. Each person is easy to distinguish on their own.

1 = Medium: People are moderately close together. Some overlap between individuals, but gaps and open space are still visible.

2 = High: People are tightly packed with little or no visible space between them. Individuals often overlap and may be hard to distinguish separately.

Return ONLY:

Count: <integer>
Density: <0/1/2>

Do not explain.
Do not estimate.
Do not output ranges.
Return exactly one integer count.
"""
        }]
    })

    # Only this final turn differs from build_messages_zeroshot_A() above —
    # the two lines giving the YOLO count, same wording as every other Experiment B.
    conversation.append({
        "role": "user",
        "content": [
            {"type": "image", "image": load_image(test_image)},
            {"type": "text", "text": f"""TEST IMAGE

YOLO predicted count: {int(yolo_count)}

Use the YOLO predicted count as additional information, but inspect the image yourself.

Count every visible person exactly once.

Return ONLY

Count: <integer>

Density: <0/1/2>
"""}
        ]
    })

    return conversation

## 10. Inference, parsing, and verification

In [ ]:
import gc
import re
import torch


def run_gemma(conversation):
    inputs = processor.apply_chat_template(
        conversation,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        do_pan_and_scan=False,
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=40, do_sample=False)

    trimmed_ids = generated_ids[:, input_len:]
    response = processor.batch_decode(
        trimmed_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    del inputs, generated_ids, trimmed_ids
    gc.collect()
    torch.cuda.empty_cache()

    return response.strip()


def parse_response(response):
    """Rejects degenerate long digit-runs as unparseable rather than
    taking them literally (same guard used throughout this project)."""
    response = str(response)

    count = None
    m = re.search(r"Count\s*[:=]\s*(\d+)", response, re.IGNORECASE)
    if m:
        digits = m.group(1)
        if len(digits) <= 5:
            count = int(digits)

    density = None
    m = re.search(r"Density\s*[:=]\s*([012])", response, re.IGNORECASE)
    if m:
        density = int(m.group(1))
    else:
        r = response.lower()
        if "low" in r:
            density = 0
        elif "medium" in r:
            density = 1
        elif "high" in r:
            density = 2

    return count, density

## 11. Load data and merge YOLO predictions
No `support_mapping.csv` — just `test.csv` merged directly with `yolo_predictions.csv`.

In [ ]:
import pandas as pd

test_df = pd.read_csv(f"{ROOT}/test.csv")
yolo_df = pd.read_csv(f"{ROOT}/yolo_predictions.csv")

test_B = test_df.merge(yolo_df, left_on="filename", right_on="image_id", how="left")

print("Total rows:", len(test_B))
print("Missing YOLO counts:", test_B["predicted_count"].isna().sum())

n_before = len(test_B)
test_B = test_B.dropna(subset=["predicted_count"]).reset_index(drop=True)
print(f"Dropped {n_before - len(test_B)} rows with no YOLO match; {len(test_B)} usable rows remain.")

## 12. Verify B differs from A by exactly the YOLO block

In [ ]:
_sample = test_B.iloc[0]
_test_image = os.path.join(TEST_DIR, f"{int(_sample['filename']):04d}.jpg")

_conv_A = build_messages_zeroshot_A(_test_image)
_conv_B = build_messages_zeroshot_B(_test_image, int(_sample["predicted_count"]))

def _texts(conv):
    out = []
    for m in conv:
        for c in m["content"]:
            if c.get("type") == "text":
                out.append(c["text"])
    return out

_texts_A, _texts_B = _texts(_conv_A), _texts(_conv_B)
print("System prompt identical:", _texts_A[0] == _texts_B[0])
print()
print("--- Test-image turn, A ---")
print(_texts_A[-1])
print("--- Test-image turn, B ---")
print(_texts_B[-1])

## 13. Smoke test — READ THIS OUTPUT CAREFULLY BEFORE RUNNING THE FULL LOOP

In [ ]:
N_SMOKE = 3

try:
    for idx in range(N_SMOKE):
        sample_i = test_B.iloc[idx]
        test_image_i = os.path.join(TEST_DIR, f"{int(sample_i['filename']):04d}.jpg")

        conversation_i = build_messages_zeroshot_B(test_image_i, int(sample_i["predicted_count"]))
        response_i = run_gemma(conversation_i)
        parsed_i = parse_response(response_i)

        print(f"--- Sample {idx} (filename={int(sample_i['filename'])}, true_count={int(sample_i['total_count'])}, yolo={int(sample_i['predicted_count'])}) ---")
        print("Response:", repr(response_i))
        print("Parsed  :", parsed_i)
        if not response_i.strip():
            print("⚠️  EMPTY RESPONSE — known float16/NaN failure signature.")
        if parsed_i[0] is None or parsed_i[1] is None:
            print("⚠️  Incomplete parse.")
        print()

except Exception as e:
    print("❌ SMOKE TEST FAILED.")
    print(f"{type(e).__name__}: {e}")
    raise

## 14. Full run — all usable test images, saved continuously to Drive

In [ ]:
from tqdm import tqdm
import gc
import time
import torch

OUT_PATH = os.path.join(RESULT_DIR, "gemma_experiment_B_zeroshot_densitydef.csv")

if "results_B_zs" not in globals():
    if os.path.exists(OUT_PATH):
        prior = pd.read_csv(OUT_PATH)
        results_B_zs = prior.to_dict("records")
        for r in results_B_zs:
            r["test_filename"] = int(r["test_filename"])
        print(f"Resumed {len(results_B_zs)} results from a previous session's CSV: {OUT_PATH}")
    else:
        results_B_zs = []

done_filenames = {r["test_filename"] for r in results_B_zs}
remaining = test_B[~test_B["filename"].astype(int).isin(done_filenames)]
print(f"Already done: {len(done_filenames)} | Remaining: {len(remaining)}")

MAX_RETRIES_PER_IMAGE = 2

for _, sample in tqdm(remaining.iterrows(), total=len(remaining)):

    test_image = os.path.join(TEST_DIR, f"{int(sample['filename']):04d}.jpg")

    conversation = build_messages_zeroshot_B(test_image, int(sample["predicted_count"]))

    response = None
    for attempt in range(1, MAX_RETRIES_PER_IMAGE + 1):
        try:
            response = run_gemma(conversation)
            break
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            print(f"\n[OOM] filename={int(sample['filename'])} "
                  f"attempt {attempt}/{MAX_RETRIES_PER_IMAGE} — clearing cache and retrying...")
            gc.collect()
            torch.cuda.empty_cache()
            time.sleep(2)

    if response is None:
        pred_count, pred_density = None, None
        response = "ERROR: CUDA out of memory after retries"
    else:
        pred_count, pred_density = parse_response(response)

    results_B_zs.append({
        "test_filename": int(sample["filename"]),
        "yolo_predicted_count": int(sample["predicted_count"]),
        "predicted_count": pred_count,
        "predicted_density": pred_density,
        "raw_response": response
    })

    pd.DataFrame(results_B_zs).to_csv(OUT_PATH, index=False)

    if len(results_B_zs) % 200 == 0:
        print(f"\n📥 Checkpoint: {len(results_B_zs)} done.")

failed = sum(1 for r in results_B_zs if r["predicted_count"] is None)
print("Finished:", len(results_B_zs), f"({failed} unparseable)")
print("Saved to:", OUT_PATH)

## 15. Evaluate — counting metrics

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

gemma_B_zs = pd.DataFrame(results_B_zs)

gemma_B_zs_eval = gemma_B_zs.merge(
    test_df[["filename", "total_count", "crowd_density"]],
    left_on="test_filename",
    right_on="filename",
    how="left"
)

print("Missing ground truth after merge:", gemma_B_zs_eval["total_count"].isna().sum())

n_before = len(gemma_B_zs_eval)
count_valid = gemma_B_zs_eval.dropna(subset=["predicted_count"]).copy()
n_dropped = n_before - len(count_valid)
print(f"Dropped {n_dropped}/{n_before} rows with no parseable count ({100*n_dropped/n_before:.1f}%)")

y_true_count = count_valid["total_count"]
y_pred_count = count_valid["predicted_count"]

mae = mean_absolute_error(y_true_count, y_pred_count)
rmse = np.sqrt(mean_squared_error(y_true_count, y_pred_count))

nonzero = count_valid["total_count"] > 0
mape = np.mean(np.abs(
    (count_valid.loc[nonzero, "total_count"] - count_valid.loc[nonzero, "predicted_count"])
    / count_valid.loc[nonzero, "total_count"]
)) * 100

print("Gemma Experiment B (Zero-Shot) — Counting")
print("===========================================")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAPE : {mape:.2f}%")
print(f"(computed on {len(count_valid)}/{n_before} images with a parseable count)")

## 16. Evaluate — density classification

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

density_valid = gemma_B_zs_eval.dropna(subset=["predicted_density"]).copy()
n_density_dropped = len(gemma_B_zs_eval) - len(density_valid)
print(f"Dropped {n_density_dropped}/{len(gemma_B_zs_eval)} rows with no parseable density ({100*n_density_dropped/len(gemma_B_zs_eval):.1f}%)")

y_true_density = density_valid["crowd_density"]
y_pred_density = density_valid["predicted_density"]

accuracy = accuracy_score(y_true_density, y_pred_density)
precision = precision_score(y_true_density, y_pred_density, average="weighted", zero_division=0)
recall = recall_score(y_true_density, y_pred_density, average="weighted", zero_division=0)
f1 = f1_score(y_true_density, y_pred_density, average="weighted", zero_division=0)

print("Gemma Experiment B (Zero-Shot) — Density")
print("==========================================")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

cm = confusion_matrix(y_true_density, y_pred_density, labels=[0, 1, 2])
print()
print("Confusion matrix (rows=GT, cols=pred):")
print(cm)

## 17. Save evaluated results to Drive

In [ ]:
EVAL_PATH = os.path.join(RESULT_DIR, "gemma_experiment_B_zeroshot_densitydef_evaluated.csv")
gemma_B_zs_eval.to_csv(EVAL_PATH, index=False)

print("Saved:", len(gemma_B_zs_eval), "rows")
print("Path:", EVAL_PATH)